# 01 · Agentic AI, Tools & LangGraph

Build your first tool-using agent: web search tool, LangGraph state, nodes and edges.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AfzalManihar/Agentic-AI-Learning/blob/main/01_agentic_ai_tools_and_langgraph.ipynb)

**Covered in this notebook**

- Adding a web search tool to our agent

> Part of [Agentic AI Learning](https://github.com/AfzalManihar/Agentic-AI-Learning). Add your own API keys in Colab Secrets (key icon) before running.

#05-08-26
#Agentic AI Intro

In [ ]:
# How do we bring LLMs (ChatGPT/Claude/GenAI) into our python code
# To call ChatGPT (LLM) in our python code we need API Key
# API -> Application Programming Interface
# Using API you can send request to any website from python code and
# get response back in python code

In [ ]:
# Bring the api_key from secrets into a variable
from google.colab import userdata

my_api_key = userdata.get('groq_api')

In [ ]:
pip install langchain-groq -q # q for silent installation

In [ ]:
# How the power of models are measured?
# Number of parameters (weights and bias)
# 20 billion , 27b, 120b
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = 'openai/gpt-oss-20b',
    temperature = 0, # Creativity level of the model
    api_key = my_api_key
)

In [ ]:
response = llm.invoke('Who is the current president of USA?')
print(response.content)

As of my latest update (June 2024), the President of the United States is **Joe Biden**.


## Adding a web search tool to our agent

In [ ]:
pip install -U ddgs -q

In [ ]:
pip install langchain-groq langgraph langchain-community duckduckgo-search -q

In [ ]:
# DuckDuckGoSearch
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

/tmp/ipykernel_4264/660449161.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [ ]:
search.invoke('Latest news on India')

'4 hours ago - Three minors were arrested after videos capturing the assault went viral in India, sparking anger. Election Commissioners, Sukhbir Singh Sandhu and Vivek Joshi, called the steps on new voters \'unauthorised, illegal\'; warned against voter database access being \'centralised\' in Delhi, bypassing state officials. ... Catch real-time updates of India\'s Day 5 campaign at the 2026 Asian Games across all events with a spotlight on shooting, badminton, weightlifting and hockey contingents today. 6 hours ago - Sunita Ahuja reacts after Govinda’s Lalbaugcha Raja visit with Komal RaniVibe Box Office Day 5: Film nets Rs 1.15cr in IndiaShruti Haasan on marriage: \'I came close to marriage\'Hanuman Ansh beats Toxic: sleeper hit crosses Rs 366.45 crore\'Kate Middleton wears Princess Diana’s earrings and Queen Mother’s necklace\'The Ba***ds Of Bollywood\' Season 2: Aryan\'s show to film"We must protect our Himalayas": Sri Sri Ravi Shankar on why Himalayas need urgent protectionBittu

In [ ]:
def web_search(query: str) -> str:
    """Search the web with latest information and give the result"""

    print("⭐Web Search function called")
    return search.run(query)

In [ ]:
tools = [web_search]
llm = llm.bind_tools(tools)

In [ ]:
response = llm.invoke('What is current president of USA')
print(response)

content='' additional_kwargs={'reasoning_content': 'The user asks: "What is current president of USA". We need to answer. As of 2026, the current president is Joe Biden? Wait, Joe Biden was president until 2025? Actually Joe Biden was elected in 2020, inaugurated 2021, term 2021-2025. In 2024, the election will be held. The current president as of 2026 would be the winner of 2024 election. The 2024 election hasn\'t happened yet? Actually it\'s 2026, so the 2024 election has already occurred. The winner could be either Joe Biden if re-elected, or a new president. We need up-to-date info. We should search the web.', 'tool_calls': [{'id': 'fc_214a48b2-7793-4406-a6df-71e4ed18fe21', 'function': {'arguments': '{"query":"current president of the United States 2026"}', 'name': 'web_search'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 180, 'prompt_tokens': 130, 'total_tokens': 310, 'completion_time': 0.189889487, 'completion_tokens_details': {'reasoning_tokens

In [ ]:
print(response.content)

In [ ]:
response.tool_calls

[{'name': 'web_search',
  'args': {'query': 'current president of the United States 2026'},
  'id': 'fc_214a48b2-7793-4406-a6df-71e4ed18fe21',
  'type': 'tool_call'}]

#LangGraph

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

In [ ]:
# Define the State
# Common place for everybody to write their response
# it is a list of messages
class State(TypedDict):
    messages: Annotated[list, add_messages]
    # messages : is the list where all new message will be stored

In [ ]:
def assistant_node(state: State):
    # Get the conversation history
    current_messages = state['messages']

    # Ask the LLM to give response
    response = llm.invoke(current_messages)

    # Return and store the llm's current response
    return {'messages': [response]}

In [ ]:
#tool node build code
from langgraph.prebuilt import ToolNode

tool_node = ToolNode(tools)

In [ ]:
from typing import Literal
from langgraph.graph import END

def should_continue(state: State) -> Literal['tools', END]:
    messages = state['messages']
    last_message = messages[-1]

    if last_message.tool_calls:
        return 'tools'

    else:
        return END

In [ ]:
from langgraph.graph import StateGraph, START

# 1. Create a blank graph
builder = StateGraph(State)

In [ ]:
builder.add_node('assistant', assistant_node)
builder.add_node('tools', tool_node)

In [ ]:
builder.add_edge(START, 'assistant')

In [ ]:
builder.add_conditional_edges(
    "assistant",
    should_continue
)

In [ ]:
builder.add_edge('tools', 'assistant')

In [ ]:
graph = builder.compile()

In [ ]:
# 1. Define the user's complex request
user_query = "Who is the current president of United States?"

# 2. Run the Graph
# We stream the steps so we can see the "thinking" process live
initial_state = {"messages": [("user", user_query)]}

print(f"User: {user_query}\n")
print("--- STARTING AGENT LOOP ---")

for step in graph.stream(initial_state):
    # Print the name of the node that just finished running
    node_name = list(step.keys())[0]
    print(f"📍 Node '{node_name}' finished.")

    # Optional: Print the actual output to see what happened inside
    # print(step)

print("--- END ---")

# 3. Print the final result
final_response = graph.invoke(initial_state)
print("\n🤖 Final Answer:")
print(final_response["messages"][-1].content)

User: Who is the current president of United States?

--- STARTING AGENT LOOP ---
📍 Node 'assistant' finished.
⭐Web Search function called
📍 Node 'tools' finished.
📍 Node 'assistant' finished.
--- END ---
⭐Web Search function called

🤖 Final Answer:
**Answer:**  
The current President of the United States is **Donald J. Trump**. He is serving his second term as the 47th president, having been re‑elected in the 2024 election and inaugurated on January 20, 2025.  

**Sources**  
- Web search results indicate that Donald J. Trump is the 47th and current president since January 20, 2025【assistant to=functions.web_search<|channel|>commentary <|constrain|>json<|message|>{"query":"current president of the United States 2026"}


In [ ]:
# Should we use LLMs for calculations??
# LLMs are unreliable -> slightly different answers every time
# Acceptable in language, not in math and calculations
# Secondary reason, why waste tokens on math calculations

# Add a calculator tool (a simple python function)

# Create an AI Agent, with the web search tool and calculator tool
# Please feel free to use ChatGPT/Claude/Gemini to write code
# But make sure you first give it our code and tell it to only make changes
# Not write completely new code from scratch

In [ ]:
pip install langchain-groq langgraph langchain-community duckduckgo-search -q

In [ ]:
# DuckDuckGoSearch
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

In [ ]:
def web_search(query: str) -> str:
    """Search the web with latest information and give the result"""

    print("⭐Web Search function called")
    return search.run(query)

In [ ]:
# Count words in a text
def get_word_count(text: str) -> int:
    return len(text.split())

In [ ]:
# Calculator tool
def calculator(a: float, b: float, operation: str):
    if operation == "add":
        return a + b
    if operation == "subtract":
        return a - b
    if operation == "multiply":
        return a * b
    if operation == "divide":
        return a / b

In [ ]:
# Add both tools
tools = [web_search, calculator]
llm = llm.bind_tools(tools)

In [ ]:
# Test calculator
response = llm.invoke('What is 125 multiplied by 48?')
print(response)

content='125 multiplied by 48 equals **6,000**.' additional_kwargs={'reasoning_content': "We need to compute 125 * 48. That's 125 * 48 = 125 * (50 - 2) = 125*50 - 125*2 = 6250 - 250 = 6000. So answer: 6000."} response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 155, 'total_tokens': 232, 'completion_time': 0.08149941, 'completion_tokens_details': {'reasoning_tokens': 56}, 'prompt_time': 0.008698932, 'prompt_tokens_details': None, 'queue_time': 0.052112777, 'total_time': 0.090198342}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_80501ff3a1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0cdb5-3480-7292-89f3-00174ea7e2e7-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 155, 'output_tokens': 77, 'total_tokens': 232, 'output_token_details': {'reasoning': 56}}


In [ ]:
# Challenge 2 (Medium)
# Add a get_word_count(text: str) tool that counts words in a string.
# Then ask the agent:
# "Search for a short description of the Eiffel Tower and tell me how many words are in the result."

In [ ]:
# 1. Define the new tool
# text: str says only input string (sentences)
def get_word_count(text: str) -> int:
    """Use this tool to count words in a sentence. Count the number of words in a string"""

    print("⭐ Word Count function called")
    return len(text.split())

In [ ]:
tools = [web_search, get_word_count]
llm = llm.bind_tools(tools)

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

In [ ]:
# Define the State
# Common place for everybody to write their response
# it is a list of messages
class State(TypedDict):
    messages: Annotated[list, add_messages]
    # messages : is the list where all new message will be stored

In [ ]:
def assistant_node(state: State):
    # Get the conversation history
    current_messages = state['messages']

    # Ask the LLM to give response
    response = llm.invoke(current_messages)

    # Return and store the llm's current response
    return {'messages': [response]}

In [ ]:
from langgraph.prebuilt import ToolNode

tool_node = ToolNode(tools)

In [ ]:
from typing import Literal
from langgraph.graph import END

def should_continue(state: State) -> Literal['tools', END]:
    messages = state['messages']
    last_message = messages[-1]

    if last_message.tool_calls:
        return 'tools'

    else:
        return END

In [ ]:
from langgraph.graph import StateGraph, START

# 1. Create a blank graph
builder = StateGraph(State)

In [ ]:
from langgraph.graph import StateGraph, START

# 1. Create a blank graph
builder = StateGraph(State)

In [ ]:
builder.add_node('assistant', assistant_node)
builder.add_node('tools', tool_node)

In [ ]:
builder.add_edge(START, 'assistant')

In [ ]:
builder.add_conditional_edges(
    "assistant",
    should_continue
)

In [ ]:
builder.add_edge('tools', 'assistant')

In [ ]:
graph = builder.compile()

In [ ]:
# 1. Define the user's complex request
user_query = "Search for a short description of the Eiffel Tower and tell me how many words are in the result."

# 2. Run the Graph
# We stream the steps so we can see the "thinking" process live
initial_state = {"messages": [("user", user_query)]}

print(f"User: {user_query}\n")
print("--- STARTING AGENT LOOP ---")

for step in graph.stream(initial_state):
    # Print the name of the node that just finished running
    node_name = list(step.keys())[0]
    print(f"📍 Node '{node_name}' finished.")

    # Optional: Print the actual output to see what happened inside
    # print(step)

print("--- END ---")

# 3. Print the final result
final_response = graph.invoke(initial_state)
print("\n🤖 Final Answer:")
print(final_response["messages"][-1].content)

User: Search for a short description of the Eiffel Tower and tell me how many words are in the result.

--- STARTING AGENT LOOP ---
📍 Node 'assistant' finished.
⭐Web Search function called
📍 Node 'tools' finished.
📍 Node 'assistant' finished.
--- END ---
⭐Web Search function called

🤖 Final Answer:
**Short description (20 words)**  
The Eiffel Tower is a wrought‑iron lattice tower on the Champ‑de‑Mars in Paris, named after engineer Gustave Eiffel.  

**Word count:** 20 words.
